In [1]:
#URL : https://www.kaggle.com/competitions/playground-series-s6e3/overview

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBClassifier
import warnings

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
X_train = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/playground-series-s6e3/train.csv',index_col = 'id')
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/playground-series-s6e3/test.csv',index_col = 'id')
y_train = X_train['Churn']
X_train.drop(columns = 'Churn',inplace = True)

In [4]:
X_train.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [5]:
print([col for col in X_train.columns if X_train[col].dtype in ['int','float']])
print

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


<function print>

In [ ]:
num_cols = ['tenure','MonthlyCharges','TotalCharges']
cat_Or_cols = ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
cat_Bin_cols = ['gender','SeniorCitizen','Partner','Dependents','PhoneService','PaperlessBilling']
cat_OH_cols = ['InternetService','Contract','PaymentMethod',]

In [ ]:
def binary_map(x):
    return x.replace({
        'Yes' : 1,
        'No' : 0,
        'Male' : 1,
        'Female' : 0,
        1 : 1,
        0 : 0
    })
Binary_Encoder = FunctionTransformer(binary_map,validate = False)

preprocessing = ColumnTransformer(transformers = [
    ('num',StandardScaler(copy = True,with_mean = True,with_std = True),num_cols),
    ('cat_binary',Binary_Encoder,cat_Bin_cols),
    ('cat_Or',OrdinalEncoder(dtype = int,handle_unknown = 'use_encoded_value',unknown_value = -1),cat_Or_cols),
    ('cat_Oh',OneHotEncoder(categories = 'auto',sparse_output = True,handle_unknown = 'ignore'),cat_OH_cols)
])
y_train = y_train.map({'Yes' : 1,'No' : 0})

In [ ]:
model = Pipeline(steps = [
    ('preprocess',preprocessing),
    ('model',XGBClassifier(
        tree_method='hist',
        device='cuda',
        
        n_estimators = 100,
        learning_rate = 0.05,
        reg_alpha = 0,
        reg_lambda = 1,
    ))
])

In [ ]:
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv = kf,
    scoring = 'neg_log_loss'
)
print(-scores.mean())

In [ ]:
model.fit(X_train,y_train)

In [ ]:
preds = model.predict(X_test)
submission = pd.DataFrame({
    'id' : X_test.index,
    'Churn' : preds
})
submission.to_csv('submission.csv',index = False)